# Feature Engineering

In [1]:
import sys
print(sys.executable)

/home/zocyus/Documents/KaggleGettingStarted/.venv/bin/python


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
CLEAN_DATA_DIR = DATA_DIR / "data_clean"

In [ ]:
# NOT THE IDEAL, BUT IT WORKS, AFTER, SEARCH FOR ANOTHER APPROACH ON cleaning.py

df_clean = pd.read_csv(CLEAN_DATA_DIR / "train_clean.csv", keep_default_na=False, na_values=[""])

## Classification

In [ ]:
df_clean.head()

In [ ]:
df_clean.info()

In [ ]:
qual_ord = [
    "LOTSHAPE", "UTILITIES", "LANDSLOPE", "OVERALLQUAL",
    "OVERALLCOND", "EXTERQUAL", "EXTERCOND", "BSMTQUAL",
    "BSMTCOND", "BSMTEXPOSURE", "BSMTFINTYPE1", "BSMTFINTYPE2",
    "HEATINGQC", "ELECTRICAL", "KITCHENQUAL", "FUNCTIONAL",
    "FIREPLACEQU", "GARAGEFINISH", "GARAGEQUAL", "GARAGECOND",
    "PAVEDDRIVE", "FENCE"
]

qual_nom = [
    "MSSUBCLASS", "MSZONING", "STREET", "ALLEY", "LANDCONTOUR",
    "LOTCONFIG", "NEIGHBORHOOD", "CONDITION1", "CONDITION2",
    "BLDGTYPE", "HOUSESTYLE", "ROOFSTYLE", "ROOFMATL",
    "EXTERIOR1ST", "EXTERIOR2ND", "MASVNRTYPE", "FOUNDATION",
    "HEATING", "CENTRALAIR", "GARAGETYPE", "MISCFEATURE",
    "SALETYPE", "SALECONDITION"
]


quant_disc = df_clean.select_dtypes(include='int64').drop(
    columns=["ID"] + qual_ord + qual_nom, errors="ignore"
)

quant_cont = df_clean.select_dtypes(include='float64')
quant_bool = df_clean.select_dtypes(include='bool')


In [ ]:
df_qual_ord = df_clean.copy()

for col in qual_ord:
    print(df_qual_ord[col].value_counts().sort_index(ascending=False))
    print()

In [ ]:
columns_to_drop = ["UTILITIES"]
columns_to_add = []

# GRADES 1-10
mapping_1_10_grades = {
    0: 1,
    1: 1,
    2: 1,
    3: 1,
    4: 1,
    5: 2,
    6: 2,
    7: 2,
    8: 3,
    9: 3,
    10: 3,
}
df_qual_ord["OVERALLCOND"] = df_qual_ord["OVERALLCOND"].map(mapping_1_10_grades)
df_qual_ord["OVERALLQUAL"] = df_qual_ord["OVERALLQUAL"].map(mapping_1_10_grades)

# FIVE QUALITIES
map_qual_5 = {
    "NA": 0,
    "PO": 0,
    "FA": 2,
    "TA": 2,
    "GD": 3,
    "EX": 3,
}
df_qual_ord["EXTERQUAL"] = df_qual_ord["EXTERQUAL"].map(map_qual_5)
df_qual_ord["EXTERCOND"] = df_qual_ord["EXTERCOND"].map(map_qual_5)
df_qual_ord["HEATINGQC"] = df_qual_ord["HEATINGQC"].map(map_qual_5)
df_qual_ord["KITCHENQUAL"] = df_qual_ord["KITCHENQUAL"].map(map_qual_5)

# SIX QUALITIES
map_qual_6 = {"NA": 0, "PO": 0, "FA": 2, "TA": 2, "GD": 3, "EX": 3}

df_qual_ord["BSMTQUAL"] = df_qual_ord["BSMTQUAL"].map(map_qual_6)
df_qual_ord["BSMTCOND"] = df_qual_ord["BSMTCOND"].map(map_qual_6)
df_qual_ord["FIREPLACEQU"] = df_qual_ord["FIREPLACEQU"].map(map_qual_6)
df_qual_ord["GARAGEQUAL"] = df_qual_ord["GARAGEQUAL"].map(map_qual_6)
df_qual_ord["GARAGECOND"] = df_qual_ord["GARAGECOND"].map(map_qual_6)

# BSMT TYPE
map_bsmt_fin = {
    "NA": 0,
    "UNF": 1,
    "LWQ": 2,
    "REC": 2,
    "BLQ": 2,
    "ALQ": 3,
    "GLQ": 3,
}
df_qual_ord["BSMTFINTYPE1"] = df_qual_ord["BSMTFINTYPE1"].map(map_bsmt_fin)
df_qual_ord["BSMTFINTYPE2"] = df_qual_ord["BSMTFINTYPE2"].map(map_bsmt_fin)

# SPECIAL CASES
df_qual_ord["LANDSLOPE"] = df_qual_ord["LANDSLOPE"].map(
    {"NA": 0, "SEV": 3, "MOD": 2, "GTL": 1}
)
df_qual_ord["BSMTEXPOSURE"] = df_qual_ord["BSMTEXPOSURE"].map(
    {"NA": 0, "NO": 1, "MN": 1, "AV": 1, "GD": 2}
)
df_qual_ord["GARAGEFINISH"] = df_qual_ord["GARAGEFINISH"].map(
    {"NA": 0, "UNF": 1, "RFN": 2, "FIN": 3}
)

df_qual_ord["ELECTRICAL"] = df_qual_ord["ELECTRICAL"].map(
    {"MIX": 1, "FUSEP": 1, "FUSEF": 2, "FUSEA": 2, "SBRKR": 3}
)

df_qual_ord["FENCE"] = df_qual_ord["FENCE"].map(
    {"NA": 0, "MNWW": 1, "GDWO": 1, "MNPRV": 1, "GDPRV": 1}
)

df_qual_ord["FUNCTIONAL"] = df_qual_ord["FUNCTIONAL"].map(
    {
        "SEV": 0,
        "MAJ2": 0,
        "MAJ1": 0,
        "MOD": 1,
        "MIN2": 1,
        "MIN1": 1,
        "TYP": 2,
        "NA": 0,
    }
)

# BOOLEAN CONVERSION
df_qual_ord = df_qual_ord.rename(columns={"FENCE": "HASFENCE"})
df_qual_ord["HASFENCE"] = df_qual_ord["HASFENCE"].astype("bool")
columns_to_drop.append("FENCE")
columns_to_add.append("HASFENCE")

df_qual_ord["PAVEDDRIVE"] = df_qual_ord["PAVEDDRIVE"].map(
    {"NA": 0, "MNWW": 1, "GDWO": 1, "MNPRV": 1, "GDPRV": 1}
)
df_qual_ord = df_qual_ord.rename(columns={"PAVEDDRIVE": "HASPAVEDDRIVE"})
df_qual_ord["HASPAVEDDRIVE"] = df_qual_ord["HASPAVEDDRIVE"].astype("bool")
columns_to_drop.append("PAVEDDRIVE")
columns_to_add.append("HASPAVEDDRIVE")


df_qual_ord["LOTSHAPE"] = df_qual_ord["LOTSHAPE"].map(
    {"REG": 1, "IR1": 0, "IR2": 0, "IR3": 0}
)
df_qual_ord = df_qual_ord.rename(columns={"LOTSHAPE": "HASREGULARLOTSHAPE"})
df_qual_ord["HASREGULARLOTSHAPE"] = df_qual_ord["HASREGULARLOTSHAPE"].astype("bool")
columns_to_drop.append("LOTSHAPE")
columns_to_add.append("HASREGULARLOTSHAPE")

# UPDATE QUAL_ORD
columns_to_drop
qual_ord = list((set(qual_ord) - set(columns_to_drop)))
qual_ord = list(set(qual_ord).union(set(columns_to_add)))


In [ ]:
for col in qual_ord:
    print(df_qual_ord[col].value_counts().sort_index(ascending=False))
    print()

In [ ]:
pd.set_option('display.max_columns', None)
df_qual_ord.head()

## Quantitative Discrete

In [ ]:
df_quan_disc = df_qual_ord.copy()

In [ ]:


for col in quant_disc:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histograma
    axes[0].hist(df_quan_disc[col])
    axes[0].set_title(f'Histograma - {col}')
    
    # Violinplot (troque 'outra_coluna' pela variável contínua que quer comparar)
    sns.violinplot(x=df_quan_disc[col], y=df_quan_disc['outra_coluna'], ax=axes[1])
    axes[1].set_title(f'Violinplot - {col}')
    
    plt.tight_layout()
    plt.show()

## Qualitative nominal

In [ ]:
df_qual_nom = df_quan_disc.copy()

for col in qual_nom:
    print(df_qual_nom[col].value_counts().sort_index(ascending=False))
    print()


In [ ]:
for col in qual_nom:
    df_group_by_saleprice = df_qual_nom.groupby(col)['SALEPRICE'].agg(['mean', 'count'])
    df_group_by_saleprice.columns = df_group_by_saleprice.columns.str.upper()
    df_group_by_saleprice = df_group_by_saleprice.rename(columns={'MEAN':'SALEPRICE_MEAN'}) 
    df_group_by_saleprice['SALEPRICE_MEAN'] = df_group_by_saleprice['SALEPRICE_MEAN'].round(2)
    df_group_by_saleprice = df_group_by_saleprice.sort_values(by='SALEPRICE_MEAN', ascending=False)
    df_group_by_saleprice['PCT_COUNT'] = round((df_group_by_saleprice['COUNT'] / sum(df_group_by_saleprice['COUNT']))*100,2)
    df_group_by_saleprice['PCT_DIFF'] = df_group_by_saleprice['SALEPRICE_MEAN'].pct_change().round(2)*100
    df_group_by_saleprice['PCT_DIFF'] = df_group_by_saleprice['PCT_DIFF'].fillna(0)

    df_group_by_saleprice = df_group_by_saleprice[['SALEPRICE_MEAN', 'PCT_DIFF', 'COUNT', 'PCT_COUNT']]
    print(df_group_by_saleprice)
    print("\n", 50*"-", "\n")

In [ ]:
columns_to_drop = []
columns_to_add = []


# CONVERT TO BOOL
df_qual_nom = df_qual_nom.rename(columns={"CENTRALAIR": "HASCENTRALAIR"})
df_qual_nom['HASCENTRALAIR'] = np.where(df_qual_nom['HASCENTRALAIR']=='Y', 1, 0).astype('bool')
columns_to_drop.append("CENTRALAIR")
columns_to_add.append("HASCENTRALAIR")

df_qual_nom = df_qual_nom.rename(columns={"ALLEY": "HASALLEY"})
df_qual_nom['HASALLEY'] = np.where(df_qual_nom['HASALLEY']=='NA', 0, 1).astype('bool')
columns_to_drop.append("ALLEY")
columns_to_add.append("HASALLEY")

df_qual_nom = df_qual_nom.rename(columns={"STREET": "ISPAVED"})
df_qual_nom['ISPAVED'] = np.where(df_qual_nom['ISPAVED']=='PAVE', 1, 0).astype('bool')
columns_to_drop.append("STREET")
columns_to_add.append("ISPAVED")

df_qual_nom = df_qual_nom.rename(columns={"LANDCONTOUR": "FLATNESS"})
df_qual_nom["FLATNESS"] = df_qual_nom["FLATNESS"].map({'HLS': 2, 'LOW': 0, 'LVL': 1, 'BNK': 1})
columns_to_drop.append("LANDCONTOUR")
columns_to_add.append("FLATNESS")

# UPDATE QUAL_NORM
columns_to_drop
qual_nom = list((set(qual_nom) - set(columns_to_drop)))
qual_nom = list(set(qual_nom).union(set(columns_to_add)))

